In [ ]:
   !pip install wfdb numpy scipy scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.0 MB/s eta 0:00:00


In [ ]:
from preprocess import build_dataset
X, y, patient_ids = build_dataset()


Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

In [ ]:
import numpy as np
print(X.shape, y.shape, patient_ids.shape)
print(np.unique(patient_ids))

(99536, 250) (99536,) (99536,)
[100 101 103 105 106 108 109 111 112 113 114 115 116 117 118 119 121 122
 123 124 200 201 202 203 205 207 208 209 210 212 213 214 215 219 220 221
 222 223 228 230 231 232 233 234]


In [ ]:
from google.colab import files
files.download('X.npy')
files.download('y.npy')
files.download('patient_ids.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split

num_classes = len(np.unique(y))
X_reshaped = X.reshape(-1, X.shape[1], 1)
y_cat = to_categorical(y, num_classes=num_classes)

X_train, X_test, y_train, y_test = train_test_split(
    X_reshaped, y_cat, test_size=0.2, stratify=y, random_state=42
)

def build_cnn(input_len, num_classes):
    model = Sequential([
        Conv1D(32, 5, activation='relu', input_shape=(input_len, 1)),
        MaxPooling1D(2),
        Conv1D(64, 5, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_cnn(X.shape[1], num_classes)
model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 36s 31ms/step - accuracy: 0.9616 - loss: 0.1372 - val_accuracy: 0.9830 - val_loss: 0.0566
Epoch 2/10
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 36s 32ms/step - accuracy: 0.9819 - loss: 0.0693 - val_accuracy: 0.9864 - val_loss: 0.0476
Epoch 3/10
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 35s 32ms/step - accuracy: 0.9835 - loss: 0.0585 - val_accuracy: 0.9844 - val_loss: 0.0468
Epoch 4/10
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 35s 31ms/step - accuracy: 0.9852 - loss: 0.0509 - val_accuracy: 0.9907 - val_loss: 0.0316
Epoch 5/10
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 38s 34ms/step - accuracy: 0.9878 - loss: 0.0449 - val_accuracy: 0.9912 - val_loss: 0.0311
Epoch 6/10
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 39s 35ms/step - accuracy: 0.9883 - loss: 0.0407 - val_accuracy: 0.9913 - val_loss: 0.0305
Epoch 7/10
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 42s 36ms/step - accuracy: 0.9893 - loss: 0.0364 - val_accuracy: 0.9901 - val_loss: 0.0348
Epoch 8/10
1120/1120 ━━━━━━━━━━━━━━━━━━━━ 40s 36ms/step - accuracy: 0.9896 -

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test).argmax(axis=1)
y_true = y_test.argmax(axis=1)

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

623/623 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     14953
           1       0.99      1.00      0.99      3066
           2       0.97      0.84      0.90       509
           3       0.98      0.97      0.98      1380

    accuracy                           0.99     19908
   macro avg       0.98      0.95      0.97     19908
weighted avg       0.99      0.99      0.99     19908

[[14903    18    14    18]
 [    7  3054     0     5]
 [   78     3   427     1]
 [   28     8     0  1344]]


In [ ]:
DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,
       207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,
       221,222,228,231,232,233,234]

train_mask = np.isin(patient_ids, DS1)
test_mask = np.isin(patient_ids, DS2)

X_train2 = X_reshaped[train_mask]
y_train2 = y_cat[train_mask]
X_test2 = X_reshaped[test_mask]
y_test2 = y_cat[test_mask]

# confirm no patient overlap
assert set(patient_ids[train_mask]) & set(patient_ids[test_mask]) == set()
print("No overlap confirmed. Train:", X_train2.shape, "Test:", X_test2.shape)

No overlap confirmed. Train: (50341, 250, 1) Test: (49195, 250, 1)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

model2 = build_cnn(X.shape[1], num_classes)

weights = compute_class_weight('balanced', classes=np.unique(y), y=y[train_mask])
class_weight = dict(enumerate(weights))

model2.fit(X_train2, y_train2, epochs=10, batch_size=64,
           validation_split=0.1, class_weight=class_weight)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
708/708 ━━━━━━━━━━━━━━━━━━━━ 27s 34ms/step - accuracy: 0.8690 - loss: 0.3469 - val_accuracy: 0.9605 - val_loss: 0.2372
Epoch 2/10
708/708 ━━━━━━━━━━━━━━━━━━━━ 25s 35ms/step - accuracy: 0.9341 - loss: 0.1806 - val_accuracy: 0.8091 - val_loss: 0.5053
Epoch 3/10
708/708 ━━━━━━━━━━━━━━━━━━━━ 41s 35ms/step - accuracy: 0.9493 - loss: 0.1370 - val_accuracy: 0.9426 - val_loss: 0.2478
Epoch 4/10
708/708 ━━━━━━━━━━━━━━━━━━━━ 40s 34ms/step - accuracy: 0.9578 - loss: 0.1072 - val_accuracy: 0.8167 - val_loss: 0.4793
Epoch 5/10
708/708 ━━━━━━━━━━━━━━━━━━━━ 25s 36ms/step - accuracy: 0.9647 - loss: 0.0932 - val_accuracy: 0.5488 - val_loss: 0.8391
Epoch 6/10
708/708 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.9651 - loss: 0.0825 - val_accuracy: 0.7658 - val_loss: 0.5313
Epoch 7/10
708/708 ━━━━━━━━━━━━━━━━━━━━ 41s 35ms/step - accuracy: 0.9693 - loss: 0.0725 - val_accuracy: 0.8562 - val_loss: 0.3887
Epoch 8/10
708/708 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.9723 - loss: 0.0612 - 

In [ ]:
y_pred2 = model2.predict(X_test2).argmax(axis=1)
y_true2 = y_test2.argmax(axis=1)

print(classification_report(y_true2, y_pred2))
print(confusion_matrix(y_true2, y_pred2))

1538/1538 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step
              precision    recall  f1-score   support

           0       0.86      0.80      0.83     36641
           1       0.01      0.00      0.00      7599
           2       0.03      0.12      0.04      1736
           3       0.42      0.91      0.58      3219

    accuracy                           0.66     49195
   macro avg       0.33      0.46      0.36     49195
weighted avg       0.67      0.66      0.66     49195

[[29199   277  6153  1012]
 [ 3459     5  1467  2668]
 [ 1177     0   212   347]
 [  181    65    37  2936]]
